In [1]:
import os
import pandas as pd, math
import numpy as np
from pyhive import presto
from datetime import datetime, date, timedelta
import warnings

# Ignore a specific warning by its type
warnings.filterwarnings("ignore")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [2]:
presto_port = '80'
# username = 'airflow-user'
username = 'biswagourab.roy@rapido.bike'

connection1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
connection2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
connection3 = presto.connect('processing-2.processing.data.production.internal', presto_port, username)
connection4 = presto.connect('processing.processing.data.production.internal', presto_port, username)

In [3]:
city = 'Chennai'
date_list = ['2024-04-07', '2024-04-14', '2024-04-21', '2024-04-28', '2024-05-05']
##, '2024-04-21', '2024-04-28', '2024-05-05']
# date_list = [ '2024-04-07', '2024-04-14', '2024-04-21', '2024-04-28', '2024-05-05']
base_week = date_list[0]

In [4]:
def consideration_query(run_date, city_name, base_week):
    
    rr_data_query = f"""
    
    with b as (
                select 
                    customerid,
                    -- sum(ao_sessions_unique_daily) ao,
                    -- sum(fe_sessions_unique_daily) fe,
                    -- sum(rr_sessions_unique_daily) rr,
                    -- sum(net_rides_daily) nr,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then ao_sessions_unique_daily end) as ao,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then net_rides_daily end) as nr,
                    
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w10,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w09,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w08,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w07,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w06,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w05,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w04,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w03,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w02,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w01,
                    
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w10,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w09,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w08,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w07,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w06,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w05,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w04,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w03,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w02,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w01,
                    
                    
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w10,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w09,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w08,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w07,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w06,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w05,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w04,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w03,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w02,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w01

                from 
                    datasets.customer_rf_daily_kpi 
                where 
                    date(day) > date('{base_week}') 
                    and date(day) <= date('{base_week}') + interval '70' day
                    and service_name in ('Link')
                    and city in ('{city_name}')
                group by 1
                --having sum(case when service_name in ('Link') then fe_sessions_unique_daily end)>=4
                ),
  
       dormant_customer as (
            
            select
                customer_id as customer,
                date_diff('day',taxi_lifetime_first_ride_date,date(run_date)) rapido_first_ride,
                ceil(cast(date_diff('day',taxi_lifetime_first_ride_date,date(run_date)) as real)/10)*10 rapido_age_seg_fr,
                
                taxi_lifetime_last_ride_date, taxi_lifetime_first_ride_date, 
                taxi_lifetime_stage, previous_taxi_lifetime_stage, taxi_income_segment,
                taxi_lifetime_rides, taxi_recency_segment, taxi_rr_active_days_last_21_days as run_taxi_rr_active_days_last_21_days,
                taxi_lifetime_last_ride_city city_name, 
                
                fe_mean_intent_type, fe_intent_trend_type, fe_potential_per_type, 
                fe_regularity_type, fe_recency_type, fe_volatility_type,
                
                gender_tag, ps_tag_link, ps_tag_auto,
                
                taxi_service_affinity, taxi_distance_preference, taxi_time_of_day_preference, taxi_day_of_week_preference
            
            from 
                datasets.iallocator_customer_segments
            where 
                date(run_date) = date('{base_week}')
                -- and taxi_lifetime_last_ride_date = date('2023-11-01')
                and taxi_recency_segment in ('RECENT')
                --and taxi_lifetime_rides < 4
                --and customer_id in (select customerid from b)
                and taxi_lifetime_last_ride_city in ('{city_name}')
        ),
        
        base as (
        
            select 
                a.*,
                case when coalesce(b.rr,0) <1 then 0 else 1 end as rr_retention_flag,
                case when coalesce(b.fe,0) <1 then 0 else 1 end as fe_retention_flag,
                case when coalesce(b.ao,0) <1 then 0 else 1 end as ao_retention_flag,
                case when coalesce(b.nr,0) <1 then 0 else 1 end as net_retention_flag,
                
                case when coalesce(b.nr_post_w01,0) <1 then 0 else 1 end as nr_flag_w01,
                case when coalesce(b.nr_post_w02,0) <1 then 0 else 1 end as nr_flag_w02,
                case when coalesce(b.nr_post_w03,0) <1 then 0 else 1 end as nr_flag_w03,
                case when coalesce(b.nr_post_w04,0) <1 then 0 else 1 end as nr_flag_w04,
                case when coalesce(b.nr_post_w05,0) <1 then 0 else 1 end as nr_flag_w05,
                case when coalesce(b.nr_post_w06,0) <1 then 0 else 1 end as nr_flag_w06,
                case when coalesce(b.nr_post_w07,0) <1 then 0 else 1 end as nr_flag_w07,
                case when coalesce(b.nr_post_w08,0) <1 then 0 else 1 end as nr_flag_w08,
                case when coalesce(b.nr_post_w09,0) <1 then 0 else 1 end as nr_flag_w09,
                case when coalesce(b.nr_post_w10,0) <1 then 0 else 1 end as nr_flag_w10,
                
                case when coalesce(b.rr_post_w01,0) <1 then 0 else 1 end as rr_flag_w01,
                case when coalesce(b.rr_post_w02,0) <1 then 0 else 1 end as rr_flag_w02,
                case when coalesce(b.rr_post_w03,0) <1 then 0 else 1 end as rr_flag_w03,
                case when coalesce(b.rr_post_w04,0) <1 then 0 else 1 end as rr_flag_w04,
                case when coalesce(b.rr_post_w05,0) <1 then 0 else 1 end as rr_flag_w05,
                case when coalesce(b.rr_post_w06,0) <1 then 0 else 1 end as rr_flag_w06,
                case when coalesce(b.rr_post_w07,0) <1 then 0 else 1 end as rr_flag_w07,
                case when coalesce(b.rr_post_w08,0) <1 then 0 else 1 end as rr_flag_w08,
                case when coalesce(b.rr_post_w09,0) <1 then 0 else 1 end as rr_flag_w09,
                case when coalesce(b.rr_post_w10,0) <1 then 0 else 1 end as rr_flag_w10,
                
                case when coalesce(b.fe_post_w01,0) <1 then 0 else 1 end as fe_flag_w01,
                case when coalesce(b.fe_post_w02,0) <1 then 0 else 1 end as fe_flag_w02,
                case when coalesce(b.fe_post_w03,0) <1 then 0 else 1 end as fe_flag_w03,
                case when coalesce(b.fe_post_w04,0) <1 then 0 else 1 end as fe_flag_w04,
                case when coalesce(b.fe_post_w05,0) <1 then 0 else 1 end as fe_flag_w05,
                case when coalesce(b.fe_post_w06,0) <1 then 0 else 1 end as fe_flag_w06,
                case when coalesce(b.fe_post_w07,0) <1 then 0 else 1 end as fe_flag_w07,
                case when coalesce(b.fe_post_w08,0) <1 then 0 else 1 end as fe_flag_w08,
                case when coalesce(b.fe_post_w09,0) <1 then 0 else 1 end as fe_flag_w09,
                case when coalesce(b.fe_post_w10,0) <1 then 0 else 1 end as fe_flag_w10

            from 
                dormant_customer a
            left join b
                
                on a.customer=b.customerid

        ),
    
        fe_data as (
        
            select 
                fe.*
            from
                (
                select 
                    aa.*, 
                    bb.fe_LT, 
                    bb.rapido_age,
                    bb.rapido_age_seg
                from
                    (
                    select 
                        customerid,
                        sum(case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_w13,
                        sum(case when day > date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_w12,
                        sum(case when day > date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_w11,
                        sum(case when day > date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_w10,
                        sum(case when day > date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_w09,
                        sum(case when day > date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_w08,
                        sum(case when day > date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_w07,
                        sum(case when day > date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_w06,
                        sum(case when day > date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_w05,
                        sum(case when day > date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_w04,
                        sum(case when day > date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_w03,
                        sum(case when day > date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_w02,
                        sum(case when day > date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_w01,
                        sum(case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_91days,
                        sum(case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_91days,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d')  then day end) as active_fe_days_w13,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d')  then day end) as active_fe_days_w12,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d')  then day end) as active_fe_days_w11,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d')  then day end) as active_fe_days_w10,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d')  then day end) as active_fe_days_w09,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d')  then day end) as active_fe_days_w08,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d')  then day end) as active_fe_days_w07,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d')  then day end) as active_fe_days_w06,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d')  then day end) as active_fe_days_w05,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d')  then day end) as active_fe_days_w04,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d')  then day end) as active_fe_days_w03,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d')  then day end) as active_fe_days_w02,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then day end) as active_fe_days_w01,
                        day((date('{run_date}') + interval '-01' day) - date(max(day))) as days_since_last_fe
                    from 
                        (
                        select 
                            customerid,day, service_name,
                            sum(fe_sessions_unique_daily) as fe_sessions_unique_daily,  
                            sum(rr_sessions_unique_daily) as rr_sessions_unique_daily
                        from 
                            datasets.customer_rf_daily_kpi a
                        join
                            base b
                            on a.customerid = b.customer
                            and 1 = 1 
                            and city = '{city_name}'
                            and fe_sessions_unique_daily >= 1 
                            and day >= date_format(date('{run_date}') + interval '-91' day , '%Y-%m-%d')
                            and day <= date_format(date('{run_date}') + interval '-1' day , '%Y-%m-%d')
                            and service_name in ('Link')
                        group by customerid, day,service_name
                        )
                    group by customerid
                    having sum(case when service_name in ('Link') then fe_sessions_unique_daily end)>=4
                    ) aa 
                left join 
                    (
                    select 
                        customerid, 
                        sum(fe_sessions_unique_daily) as fe_LT , 
                        day((date('{run_date}') + interval '-01' day) - date(min(day))) as rapido_age,
                        ceil(cast(day((date('{run_date}') + interval '-01' day) - date(min(day))) as real)/10)*10 rapido_age_seg
                    from 
                        datasets.customer_rf_daily_kpi a
                    join
                        base b
                        on a.customerid=b.customer
                        and fe_sessions_unique_daily >= 1 
                        and city = '{city_name}'
                        and service_name = 'Link'
                    group by 1
                    having sum(case when service_name in ('Link') then fe_sessions_unique_daily end)>=4
                    ) bb
                    on aa.customerid = bb.customerid
                ) fe 
        )
        
        select 
            a.*, b.*,
            '{run_date}' as run_date
        from 
            base a
        left join
            fe_data b
            on a.customer=b.customerid
    """

    try:
        data = pd.read_sql(rr_data_query, connection3)
    except:
        data = pd.read_sql(rr_data_query, connection4)
        
    
    ## Data fetched successfully
    print(data.shape)


    ## .mask(data[columns_to_replace] == 0, pd.NA) replaces all 0 values in these columns with pd.NA.
    columns_to_replace = ['active_fe_days_w13', 'active_fe_days_w12', 'active_fe_days_w11','active_fe_days_w10', 'active_fe_days_w09',
                          'active_fe_days_w08', 'active_fe_days_w07', 'active_fe_days_w06', 'active_fe_days_w05', 'active_fe_days_w04',
                          'active_fe_days_w03', 'active_fe_days_w02', 'active_fe_days_w01']
    data[columns_to_replace] = data[columns_to_replace].mask(data[columns_to_replace] == 0, pd.NA)
    

    ## Unused
    ## FE-Normalization = Total no. of FE/Active days
    """
    data['fe_norm_w13'] = data.fe_w13/data.active_fe_days_w13
    data['fe_norm_w12'] = data.fe_w12/data.active_fe_days_w12
    data['fe_norm_w11'] = data.fe_w11/data.active_fe_days_w11
    data['fe_norm_w10'] = data.fe_w10/data.active_fe_days_w10
    data['fe_norm_w09'] = data.fe_w09/data.active_fe_days_w09
    data['fe_norm_w08'] = data.fe_w08/data.active_fe_days_w08
    data['fe_norm_w07'] = data.fe_w07/data.active_fe_days_w07
    data['fe_norm_w06'] = data.fe_w06/data.active_fe_days_w06
    data['fe_norm_w05'] = data.fe_w05/data.active_fe_days_w05
    data['fe_norm_w04'] = data.fe_w04/data.active_fe_days_w04
    data['fe_norm_w03'] = data.fe_w03/data.active_fe_days_w03
    data['fe_norm_w02'] = data.fe_w02/data.active_fe_days_w02
    data['fe_norm_w01'] = data.fe_w01/data.active_fe_days_w01
    """
    
    raw_customer_data = data[data['customerid']!=0]
    back_up = raw_customer_data.copy() ## With NA as value
    fe_data2 = raw_customer_data.fillna(0)
    fe_data = fe_data2[fe_data2['customerid']!=0] ## With 0 as value
    
    
    ## Consistency/Regularity Threshold 
    ## FE Regularity → Count of active weeks in last 13 weeks/ (13 or Rapido age in weeks if a customer is less than 13 weeks old)
    ## Active Days Mean → Total active days in last 13 weeks/ count of active weeks in last 13 weeks
    
    def regularity_decider_v2(x,active_mean):
        
        if x<=0.2:
            return '05_rare_need'
        elif x>0.2 and x<=0.4:
            return '04_monthly'
        elif x>0.4 and x<=0.689:
            return '03_bi_weekly'
        else:
            if active_mean>2.6:
                return '01_daily'
            else:
                return '02_weekly'


    ## Intent Threshold
    ## Intent → Total FE in last 13 weeks/ No. of Active weeks
    def intent_decider(x):
        
        if x<=2:
            return '03_Low'
        elif x>2 and x<=5:
            return '02_Medium'
        else:
            return '01_High'
    

    ## Consideration signal
    def consideration_signal_creator(x,y,active_mean):
        
        if y<=1.5:
            if x<=0.3:
                return 'rarely_1_rr'
            elif x>0.3 and x<=0.5:
                return 'once_in_4_weeks_1_rr'
            elif x>0.5 and x<=0.75:
                return 'once_in_2_weeks_1_rr'
            else:
                return 'weekly_1_rr'
        else:
            if x<=0.3:
                return 'rarely_2+_rr'
            elif x>0.3 and x<=0.5:
                return 'once_in_4_weeks_2+_rr'
            elif x>0.5 and x<=0.75:
                return 'once_in_2_weeks_2+_rr'
            else:
                return 'weekly_2+_rr'
                

    ## Correction factor
    def correction_factor(fe_weight,rapido_age):
        
        if rapido_age >= 91:
            week_inactive = 0
            multiplier = 1

        elif rapido_age == 0:
            week_inactive = (91-rapido_age)//7 - 1
            multiplier = (fe_weight.sum()/(fe_weight.sum()- (week_inactive*(week_inactive+1)/2)))

        # 7,14,21,28,...
        elif rapido_age%7 == 0:
            week_inactive = (91-rapido_age)//7 - 1
            multiplier = (fe_weight.sum()/(fe_weight.sum()- (week_inactive*(week_inactive+1)/2)))    
            
        else:
            week_inactive = (91-rapido_age)//7
            multiplier = (fe_weight.sum()/(fe_weight.sum()- (week_inactive*(week_inactive+1)/2))) 
        
        return (multiplier, 
                week_inactive, 
                (fe_weight.sum()- (week_inactive*(week_inactive+1)/2)), 
                ((fe_weight*fe_weight).sum() - (week_inactive*(week_inactive+1)*(2*week_inactive+1)/6))
               )

    
    ## Recency trend
    def recency_type(days_since_last_fe):
        
        if days_since_last_fe <= 6: 
            return('Recent')
        elif days_since_last_fe <= 20:
            return('Stationary')
        else:
            return('Dormant')

    ## Intent trend
    def intent_trend_type(intent_trend):
        
        if intent_trend <= -0.1:
            return('03_Declining')
        elif intent_trend <= 0.1:
            return('02_Stable')
        else:
            return('01_Increasing')
    
    
    ## Creating Final signal
    def create_L3_signals(fe_data, back_up): 
        
        print('Creating L3 Signal')
        fe_weight = np.array([1,2,3,4,5,6,7,8,9,10,11,12,13])

        ## sum (rr_sessions_unique_daily)
        fe_cols = ['fe_w13', 'fe_w12', 'fe_w11', 'fe_w10', 'fe_w09','fe_w08', 'fe_w07', 
                   'fe_w06','fe_w05', 'fe_w04', 'fe_w03', 'fe_w02','fe_w01']   
        ## count (rr_days)
        activedays_cols = ['active_fe_days_w13', 'active_fe_days_w12', 'active_fe_days_w11', 'active_fe_days_w10', 
                           'active_fe_days_w09','active_fe_days_w08', 'active_fe_days_w07', 'active_fe_days_w06',
                           'active_fe_days_w05', 'active_fe_days_w04', 'active_fe_days_w03','active_fe_days_w02', 
                           'active_fe_days_w01'] 
        
        
        print('Calctn-1')
        fe_data['fe_intent_multiplier'] = fe_data['rapido_age'].apply(lambda x: correction_factor(fe_weight,x)[0])
        
        fe_data['fe_mean_intent'] =  back_up[fe_cols].mean(axis=1) ## Check logic once
        fe_data['fe_mean_intent_final'] = fe_data['fe_mean_intent'].apply(lambda x: intent_decider(x))

        fe = fe_data[fe_cols]
        fe = fe.ge(1.0).astype(int)
        
        # This multiplies the sum of the weighted values by 1/91 to normalize the result.
        # rr_w13 to rr_w01 || rr_weight 1 to 13
        fe_data['fe_regularity'] = (1/91)*(np.array(fe)*fe_weight).sum(axis=1)
        fe_data['fe_regularity_scaled'] = fe_data['fe_regularity']*fe_data['fe_intent_multiplier']
        
        fe_data['active_days_mean'] =  back_up[activedays_cols].mean(axis=1) ## Check logic once
        
        print('Calctn-2')
        fe_data['fe_regularity_final'] = fe_data.apply(lambda x: regularity_decider_v2(x['fe_regularity_scaled'],x['active_days_mean']),axis=1)
        fe_data['reg_intent'] = fe_data.fe_regularity_final+'_'+fe_data.fe_mean_intent_final
        
        return(fe_data)

    
    print('UDF-Created Successfully')
    
    
    L3_signals = create_L3_signals(fe_data,back_up)
    L3_signals['city'] = city_name
    
    final_data = L3_signals.copy()
    
    return final_data    

In [5]:
concatenated_df=pd.DataFrame()

for i in date_list:
    
    print(i)
    tdf = consideration_query(i, city, base_week)
    
    print(tdf.shape[0])
    concatenated_df=pd.concat([concatenated_df, tdf]).reset_index(drop=True)

2024-04-07
(538211, 93)
UDF-Created Successfully
Creating L3 Signal
Calctn-1
Calctn-2
444877
2024-04-14
(538211, 93)
UDF-Created Successfully
Creating L3 Signal
Calctn-1
Calctn-2
456516
2024-04-21
(538211, 93)
UDF-Created Successfully
Creating L3 Signal
Calctn-1
Calctn-2
461033
2024-04-28
(538211, 93)
UDF-Created Successfully
Creating L3 Signal
Calctn-1
Calctn-2
464644
2024-05-05
(538211, 93)
UDF-Created Successfully
Creating L3 Signal
Calctn-1
Calctn-2
466892


In [6]:
concatenated_df.fe_regularity_scaled.quantile([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0])

0.00    0.010989
0.10    0.285714
0.20    0.384615
0.30    0.472527
0.40    0.549451
0.50    0.637363
0.60    0.717647
0.70    0.802198
0.80    0.890110
0.90    1.000000
0.95    1.000000
0.99    1.000000
1.00    1.000000
Name: fe_regularity_scaled, dtype: float64

In [7]:
concatenated_df.active_days_mean.quantile([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0])

0.00    1.000000
0.10    1.090909
0.20    1.285714
0.30    1.400000
0.40    1.555556
0.50    1.727273
0.60    2.000000
0.70    2.200000
0.80    2.615385
0.90    3.400000
0.95    4.166667
0.99    5.461538
1.00    7.000000
Name: active_days_mean, dtype: float64

In [8]:
concatenated_df.fe_mean_intent.quantile([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0])

0.00      1.000000
0.10      1.500000
0.20      1.800000
0.30      2.000000
0.40      2.375000
0.50      2.700000
0.60      3.111111
0.70      3.666667
0.80      4.500000
0.90      6.000000
0.95      7.909091
0.99     12.076923
1.00    146.666667
Name: fe_mean_intent, dtype: float64

In [27]:
concatenated_df.head(5)

,customer,rapido_first_ride,rapido_age_seg_fr,taxi_lifetime_last_ride_date,taxi_lifetime_first_ride_date,taxi_lifetime_stage,previous_taxi_lifetime_stage,taxi_income_segment,taxi_lifetime_rides,taxi_recency_segment,run_taxi_rr_active_days_last_21_days,city_name,fe_mean_intent_type,fe_intent_trend_type,fe_potential_per_type,fe_regularity_type,fe_recency_type,fe_volatility_type,gender_tag,ps_tag_link,ps_tag_auto,taxi_service_affinity,taxi_distance_preference,taxi_time_of_day_preference,taxi_day_of_week_preference,rr_retention_flag,fe_retention_flag,ao_retention_flag,net_retention_flag,nr_flag_w01,nr_flag_w02,nr_flag_w03,nr_flag_w04,nr_flag_w05,nr_flag_w06,nr_flag_w07,nr_flag_w08,nr_flag_w09,nr_flag_w10,rr_flag_w01,rr_flag_w02,rr_flag_w03,rr_flag_w04,rr_flag_w05,rr_flag_w06,rr_flag_w07,rr_flag_w08,rr_flag_w09,rr_flag_w10,fe_flag_w01,fe_flag_w02,fe_flag_w03,fe_flag_w04,fe_flag_w05,fe_flag_w06,fe_flag_w07,fe_flag_w08,fe_flag_w09,fe_flag_w10,customerid,fe_w13,fe_w12,fe_w11,fe_w10,fe_w09,fe_w08,fe_w07,fe_w06,fe_w05,fe_w04,fe_w03,fe_w02,fe_w01,fe_91days,fe_91days,active_fe_days_w13,active_fe_days_w12,active_fe_days_w11,active_fe_days_w10,active_fe_days_w09,active_fe_days_w08,active_fe_days_w07,active_fe_days_w06,active_fe_days_w05,active_fe_days_w04,active_fe_days_w03,active_fe_days_w02,active_fe_days_w01,days_since_last_fe,fe_LT,rapido_age,rapido_age_seg,run_date,fe_intent_multiplier,fe_mean_intent,fe_mean_intent_final,fe_regularity,fe_regularity_scaled,active_days_mean,fe_regularity_final,reg_intent,city,reg_rank,int_rank,final_rank
0,65ced32d1ce2ab75cade20b5,46,50.0,2024-04-06,2024-02-21,COMMITTED,SUSTENANCE,MEDIUM_INCOME,10,RECENT,5,Chennai,High,Increasing,High,High,Recent,High,FEMALE,UNKNOWN,NPS,ONLY_AUTO,MEDIUM_DISTANCE,EVENING_PEAK,UNKNOWN,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,65ced32d1ce2ab75cade20b5,0.0,0.0,0.0,0.0,0.0,1.0,9.0,1.0,3.0,9.0,3.0,6.0,10.0,42.0,42.0,0.0,0.0,0.0,0.0,0.0,1.0,4.0,1.0,2.0,4.0,2.0,2.0,5.0,0.0,103.0,50.0,50.0,2024-04-07,1.197368,5.250000,01_High,0.835165,1.000000,2.625000,01_daily,01_daily_01_High,Chennai,5,3,14
1,6247dedda060e8d6a3efe827,736,740.0,2024-04-08,2022-04-02,COMMITTED,COMMITTED,UNKNOWN,91,RECENT,15,Chennai,High,Increasing,High,High,Recent,High,FEMALE,UNKNOWN,NPS,AUTO_CAB,MEDIUM_DISTANCE,NON_PEAK,UNKNOWN,0,1,1,0,0,0,0,0,1,1,1,1,1,1,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,6247dedda060e8d6a3efe827,0.0,2.0,10.0,12.0,7.0,18.0,9.0,12.0,15.0,11.0,8.0,8.0,12.0,124.0,124.0,0.0,2.0,5.0,7.0,4.0,5.0,5.0,6.0,7.0,7.0,5.0,4.0,7.0,0.0,250.0,735.0,740.0,2024-04-07,1.000000,10.333333,01_High,0.989011,0.989011,5.333333,01_daily,01_daily_01_High,Chennai,5,3,14
2,649d7967119d961b8225fb7f,282,290.0,2024-04-04,2023-06-30,COMMITTED,COMMITTED,UNKNOWN,57,RECENT,5,Chennai,Low,Stable,High,High,Recent,High,MALE,NPS,PS,ONLY_AUTO,MEDIUM_DISTANCE,UNKNOWN,WEEKDAY,0,1,1,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,649d7967119d961b8225fb7f,3.0,2.0,4.0,1.0,3.0,3.0,3.0,2.0,5.0,1.0,2.0,1.0,3.0,33.0,33.0,2.0,2.0,2.0,1.0,1.0,1.0,3.0,1.0,4.0,1.0,2.0,1.0,2.0,2.0,190.0,282.0,290.0,2024-04-07,1.000000,2.538462,02_Medium,1.000000,1.000000,1.769231,02_weekly,02_weekly_02_Medium,Chennai,4,2,11
3,5d0f3480c2a56b449fa0ad56,1750,1750.0,2024-04-05,2019-06-23,COMMITTED,COMMITTED,UNKNOWN,235,RECENT,6,Chennai,Low,Stable,High,High,Recent,High,MALE,UNKNOWN,UNKNOWN,ONLY_LINK,UNKNOWN,UNKNOWN,WEEKDAY,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,5d0f3480c2a56b449fa0ad56,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,4.0,0.0,0.0,1.0,12.0,12.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,0.0,1.0,1.0,504.0,1310.0,1310.0,2024-04-07,1.000000,3.000000,02_Medium,0.362637,0.362637,1.750000,04_monthly,04_monthly_02_Medium,Chennai,2,2,5
4,654cd1d527b6d81a60558edd,21,30.0,2024-04-07,2024-03-17,HOOK,HANDHOLDING,MEDIUM_INCOME,5,RECENT,4,Chennai,Low,Increasing,High,High,Recent,High,FEMALE,PS,UNKNOWN,ONLY_LINK,MEDIUM_DISTANCE,EVENING_PEAK,WEEKEND,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,1,1,1

In [9]:
concatenated_df['reg_intent'] = concatenated_df.fe_regularity_final + '_' + concatenated_df.fe_mean_intent_final
base_df = concatenated_df[concatenated_df.run_date == base_week].reset_index(drop=True)

In [10]:
concatenated_df.head(3)

,customer,rapido_first_ride,rapido_age_seg_fr,taxi_lifetime_last_ride_date,taxi_lifetime_first_ride_date,taxi_lifetime_stage,previous_taxi_lifetime_stage,taxi_income_segment,taxi_lifetime_rides,taxi_recency_segment,run_taxi_rr_active_days_last_21_days,city_name,fe_mean_intent_type,fe_intent_trend_type,fe_potential_per_type,fe_regularity_type,fe_recency_type,fe_volatility_type,gender_tag,ps_tag_link,ps_tag_auto,taxi_service_affinity,taxi_distance_preference,taxi_time_of_day_preference,taxi_day_of_week_preference,rr_retention_flag,fe_retention_flag,ao_retention_flag,net_retention_flag,nr_flag_w01,nr_flag_w02,nr_flag_w03,nr_flag_w04,nr_flag_w05,nr_flag_w06,nr_flag_w07,nr_flag_w08,nr_flag_w09,nr_flag_w10,rr_flag_w01,rr_flag_w02,rr_flag_w03,rr_flag_w04,rr_flag_w05,rr_flag_w06,rr_flag_w07,rr_flag_w08,rr_flag_w09,rr_flag_w10,fe_flag_w01,fe_flag_w02,fe_flag_w03,fe_flag_w04,fe_flag_w05,fe_flag_w06,fe_flag_w07,fe_flag_w08,fe_flag_w09,fe_flag_w10,customerid,fe_w13,fe_w12,fe_w11,fe_w10,fe_w09,fe_w08,fe_w07,fe_w06,fe_w05,fe_w04,fe_w03,fe_w02,fe_w01,fe_91days,fe_91days,active_fe_days_w13,active_fe_days_w12,active_fe_days_w11,active_fe_days_w10,active_fe_days_w09,active_fe_days_w08,active_fe_days_w07,active_fe_days_w06,active_fe_days_w05,active_fe_days_w04,active_fe_days_w03,active_fe_days_w02,active_fe_days_w01,days_since_last_fe,fe_LT,rapido_age,rapido_age_seg,run_date,fe_intent_multiplier,fe_mean_intent,fe_mean_intent_final,fe_regularity,fe_regularity_scaled,active_days_mean,fe_regularity_final,reg_intent,city
0,65ced32d1ce2ab75cade20b5,46,50.0,2024-04-06,2024-02-21,COMMITTED,SUSTENANCE,MEDIUM_INCOME,10,RECENT,5,Chennai,High,Increasing,High,High,Recent,High,FEMALE,UNKNOWN,NPS,ONLY_AUTO,MEDIUM_DISTANCE,EVENING_PEAK,UNKNOWN,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,65ced32d1ce2ab75cade20b5,0.0,0.0,0.0,0.0,0.0,1.0,9.0,1.0,3.0,9.0,3.0,6.0,10.0,42.0,42.0,0.0,0.0,0.0,0.0,0.0,1.0,4.0,1.0,2.0,4.0,2.0,2.0,5.0,0.0,103.0,50.0,50.0,2024-04-07,1.197368,5.250000,01_High,0.835165,1.000000,2.625000,01_daily,01_daily_01_High,Chennai
1,6247dedda060e8d6a3efe827,736,740.0,2024-04-08,2022-04-02,COMMITTED,COMMITTED,UNKNOWN,91,RECENT,15,Chennai,High,Increasing,High,High,Recent,High,FEMALE,UNKNOWN,NPS,AUTO_CAB,MEDIUM_DISTANCE,NON_PEAK,UNKNOWN,0,1,1,0,0,0,0,0,1,1,1,1,1,1,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,6247dedda060e8d6a3efe827,0.0,2.0,10.0,12.0,7.0,18.0,9.0,12.0,15.0,11.0,8.0,8.0,12.0,124.0,124.0,0.0,2.0,5.0,7.0,4.0,5.0,5.0,6.0,7.0,7.0,5.0,4.0,7.0,0.0,250.0,735.0,740.0,2024-04-07,1.000000,10.333333,01_High,0.989011,0.989011,5.333333,01_daily,01_daily_01_High,Chennai
2,649d7967119d961b8225fb7f,282,290.0,2024-04-04,2023-06-30,COMMITTED,COMMITTED,UNKNOWN,57,RECENT,5,Chennai,Low,Stable,High,High,Recent,High,MALE,NPS,PS,ONLY_AUTO,MEDIUM_DISTANCE,UNKNOWN,WEEKDAY,0,1,1,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,649d7967119d961b8225fb7f,3.0,2.0,4.0,1.0,3.0,3.0,3.0,2.0,5.0,1.0,2.0,1.0,3.0,33.0,33.0,2.0,2.0,2.0,1.0,1.0,1.0,3.0,1.0,4.0,1.0,2.0,1.0,2.0,2.0,190.0,282.0,290.0,2024-04-07,1.000000,2.538462,02_Medium,1.000000,1.000000,1.769231,02_weekly,02_weekly_02_Medium,Chennai


In [11]:
base_df.head(3)

,customer,rapido_first_ride,rapido_age_seg_fr,taxi_lifetime_last_ride_date,taxi_lifetime_first_ride_date,taxi_lifetime_stage,previous_taxi_lifetime_stage,taxi_income_segment,taxi_lifetime_rides,taxi_recency_segment,run_taxi_rr_active_days_last_21_days,city_name,fe_mean_intent_type,fe_intent_trend_type,fe_potential_per_type,fe_regularity_type,fe_recency_type,fe_volatility_type,gender_tag,ps_tag_link,ps_tag_auto,taxi_service_affinity,taxi_distance_preference,taxi_time_of_day_preference,taxi_day_of_week_preference,rr_retention_flag,fe_retention_flag,ao_retention_flag,net_retention_flag,nr_flag_w01,nr_flag_w02,nr_flag_w03,nr_flag_w04,nr_flag_w05,nr_flag_w06,nr_flag_w07,nr_flag_w08,nr_flag_w09,nr_flag_w10,rr_flag_w01,rr_flag_w02,rr_flag_w03,rr_flag_w04,rr_flag_w05,rr_flag_w06,rr_flag_w07,rr_flag_w08,rr_flag_w09,rr_flag_w10,fe_flag_w01,fe_flag_w02,fe_flag_w03,fe_flag_w04,fe_flag_w05,fe_flag_w06,fe_flag_w07,fe_flag_w08,fe_flag_w09,fe_flag_w10,customerid,fe_w13,fe_w12,fe_w11,fe_w10,fe_w09,fe_w08,fe_w07,fe_w06,fe_w05,fe_w04,fe_w03,fe_w02,fe_w01,fe_91days,fe_91days,active_fe_days_w13,active_fe_days_w12,active_fe_days_w11,active_fe_days_w10,active_fe_days_w09,active_fe_days_w08,active_fe_days_w07,active_fe_days_w06,active_fe_days_w05,active_fe_days_w04,active_fe_days_w03,active_fe_days_w02,active_fe_days_w01,days_since_last_fe,fe_LT,rapido_age,rapido_age_seg,run_date,fe_intent_multiplier,fe_mean_intent,fe_mean_intent_final,fe_regularity,fe_regularity_scaled,active_days_mean,fe_regularity_final,reg_intent,city
0,65ced32d1ce2ab75cade20b5,46,50.0,2024-04-06,2024-02-21,COMMITTED,SUSTENANCE,MEDIUM_INCOME,10,RECENT,5,Chennai,High,Increasing,High,High,Recent,High,FEMALE,UNKNOWN,NPS,ONLY_AUTO,MEDIUM_DISTANCE,EVENING_PEAK,UNKNOWN,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,65ced32d1ce2ab75cade20b5,0.0,0.0,0.0,0.0,0.0,1.0,9.0,1.0,3.0,9.0,3.0,6.0,10.0,42.0,42.0,0.0,0.0,0.0,0.0,0.0,1.0,4.0,1.0,2.0,4.0,2.0,2.0,5.0,0.0,103.0,50.0,50.0,2024-04-07,1.197368,5.250000,01_High,0.835165,1.000000,2.625000,01_daily,01_daily_01_High,Chennai
1,6247dedda060e8d6a3efe827,736,740.0,2024-04-08,2022-04-02,COMMITTED,COMMITTED,UNKNOWN,91,RECENT,15,Chennai,High,Increasing,High,High,Recent,High,FEMALE,UNKNOWN,NPS,AUTO_CAB,MEDIUM_DISTANCE,NON_PEAK,UNKNOWN,0,1,1,0,0,0,0,0,1,1,1,1,1,1,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,6247dedda060e8d6a3efe827,0.0,2.0,10.0,12.0,7.0,18.0,9.0,12.0,15.0,11.0,8.0,8.0,12.0,124.0,124.0,0.0,2.0,5.0,7.0,4.0,5.0,5.0,6.0,7.0,7.0,5.0,4.0,7.0,0.0,250.0,735.0,740.0,2024-04-07,1.000000,10.333333,01_High,0.989011,0.989011,5.333333,01_daily,01_daily_01_High,Chennai
2,649d7967119d961b8225fb7f,282,290.0,2024-04-04,2023-06-30,COMMITTED,COMMITTED,UNKNOWN,57,RECENT,5,Chennai,Low,Stable,High,High,Recent,High,MALE,NPS,PS,ONLY_AUTO,MEDIUM_DISTANCE,UNKNOWN,WEEKDAY,0,1,1,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,649d7967119d961b8225fb7f,3.0,2.0,4.0,1.0,3.0,3.0,3.0,2.0,5.0,1.0,2.0,1.0,3.0,33.0,33.0,2.0,2.0,2.0,1.0,1.0,1.0,3.0,1.0,4.0,1.0,2.0,1.0,2.0,2.0,190.0,282.0,290.0,2024-04-07,1.000000,2.538462,02_Medium,1.000000,1.000000,1.769231,02_weekly,02_weekly_02_Medium,Chennai


In [12]:
concatenated_df.to_csv('fe_concatenated_df_old.csv')
base_df.to_csv('fe_base_df_old.csv', index=False)

In [13]:
fe_weight = np.log(np.array([3,4,5,6,7,8,9,10,11,12,13,14,15]))
fe_weight

array([1.09861229, 1.38629436, 1.60943791, 1.79175947, 1.94591015,
       2.07944154, 2.19722458, 2.30258509, 2.39789527, 2.48490665,
       2.56494936, 2.63905733, 2.7080502 ])

## Stability Function

In [14]:
def stability_func(df,base_week):
    
    df_base_week = df[df.run_date == base_week][['customer','run_date','fe_regularity_final','fe_mean_intent_final','reg_intent']]\
                                                    .reset_index(drop=True)
    ranking = {'05_rare_need' : 1,
               '04_monthly' : 2,
               '03_bi_weekly' : 3,
               '02_weekly' : 4,
               '01_daily' : 5
              }

    intent_ranking = {'03_Low' : 1,
                      '02_Medium' : 2,
                      '01_High' : 3
                     }

    final_rank = {'05_rare_need_03_Low' : 1,
                  '05_rare_need_02_Medium' : 2,
                  '05_rare_need_01_High' : 3,
                  '04_monthly_03_Low' : 4,
                  '04_monthly_02_Medium' : 5,
                  '04_monthly_01_High' : 6,
                  '03_bi_weekly_03_Low' : 7,
                  '03_bi_weekly_02_Medium' : 8,
                  '03_bi_weekly_01_High' : 9,
                  '02_weekly_03_Low' : 10,
                  '02_weekly_02_Medium' : 11,
                  '02_weekly_01_High' : 12,
                  '01_daily_02_Medium' : 13,
                  '01_daily_01_High' : 14
                 }    

    df['reg_rank'] = df['fe_regularity_final'].map(ranking)
    df['int_rank'] = df['fe_mean_intent_final'].map(intent_ranking)
    df['final_rank'] = df['reg_intent'].map(final_rank)

    
    
    ## Regularity Stabillity
    pvt_df = pd.pivot_table(df, values='reg_rank', index='customer', columns='run_date', aggfunc='sum').reset_index()
    pvt_df.columns
    dropped_reg = pvt_df.iloc[:, 1:].lt(pvt_df[base_week], axis=0)

    # Display the result
    print(dropped_reg.head())
    dropped_reg.head()
    first_dropped_week = dropped_reg.idxmax(axis=1)

    pvt_df['first_dropped_week'] = first_dropped_week.where(dropped_reg.any(axis=1), 'Maintained/Upgraded')

    pvt_df.reset_index(inplace=True)
    pvt_df.head()

    df_base_week_stability = pvt_df.merge(df_base_week, on=['customer'], how='inner')

    reg_stab = df_base_week_stability\
                        .groupby(by=['run_date','fe_regularity_final','first_dropped_week'], as_index=False)\
                        .agg(user_count=('customer','nunique'))\
                        .pivot_table(values='user_count', 
                                     index=['run_date','fe_regularity_final',], 
                                     columns='first_dropped_week', 
                                     aggfunc='sum')\
                        .reset_index()\
                        .merge(df_base_week\
                                   .groupby(by=['fe_regularity_final'],as_index=False)\
                                   .agg(base=('customer','nunique')), on=['fe_regularity_final'], how='inner')\
                        .fillna(0)
    
    # .to_csv(f'{month}/output_files/regularity_stability.csv',index=False)

    
    
    
    ## Intent Stabillity
    pvt_df_int=pd.pivot_table(df, values='int_rank', index='customer', columns='run_date', aggfunc='sum').reset_index()
    pvt_df_int.columns
    dropped_int = pvt_df_int.iloc[:, 1:].lt(pvt_df_int[base_week], axis=0)

    # Display the result
    print(dropped_int.head())
    dropped_int.head()
    int_first_dropped_week = dropped_int.idxmax(axis=1)

    pvt_df_int['int_first_dropped_week'] = int_first_dropped_week.where(dropped_int.any(axis=1), 'Maintained/Upgraded')

    pvt_df_int.reset_index(inplace=True)
    pvt_df_int.head()

    df_base_week_int_stability = pvt_df_int.merge(df_base_week, on=['customer'], how='inner')

    int_stab = df_base_week_int_stability\
                        .groupby(by=['run_date','fe_mean_intent_final','int_first_dropped_week'],as_index=False)\
                        .agg(user_count=('customer','nunique'))\
                        .pivot_table(values='user_count', 
                                     index=['run_date','fe_mean_intent_final',], 
                                     columns='int_first_dropped_week', 
                                     aggfunc='sum')\
                        .reset_index()\
                        .merge(df_base_week\
                                    .groupby(by=['fe_mean_intent_final'], as_index=False)\
                                    .agg(base=('customer','nunique')), on=['fe_mean_intent_final'], how='inner')\
                        .fillna(0)
    
    # .to_csv(f'{month}/output_files/intent_stability.csv',index=False)

    
    
    
    ## Final Stabillity
    pvt_df_final=pd.pivot_table(df, values='final_rank', index='customer', columns='run_date', aggfunc='sum').reset_index()
    pvt_df_final.columns
    dropped_final = pvt_df_final.iloc[:, 1:].lt(pvt_df_final[base_week], axis=0)

    
    # Display the result
    print(dropped_final.head())
    dropped_final.head()
    final_first_dropped_week = dropped_final.idxmax(axis=1)

    pvt_df_final['final_first_dropped_week'] = final_first_dropped_week.where(dropped_final.any(axis=1), 'Maintained/Upgraded')

    pvt_df_final.reset_index(inplace=True)
    pvt_df_final.head()

    df_base_week_final_stability = pvt_df_final.merge(df_base_week, on=['customer'], how='inner')

    final_stab = df_base_week_final_stability\
                        .groupby(by=['run_date','fe_regularity_final','fe_mean_intent_final','final_first_dropped_week'],
                                 as_index=False)\
                        .agg(user_count=('customer','nunique'))\
                        .pivot_table(values='user_count', 
                                     index=['run_date','fe_regularity_final','fe_mean_intent_final'], 
                                     columns='final_first_dropped_week', 
                                     aggfunc='sum')\
                        .reset_index()\
                        .merge(df_base_week\
                                   .groupby(by=['fe_regularity_final','fe_mean_intent_final'], as_index=False)\
                                   .agg(base=('customer','nunique')), on=['fe_regularity_final','fe_mean_intent_final'], how='inner')\
                        .fillna(0)
    
    # .to_csv(f'{month}/output_files/final_stability.csv',index=False)
    
    return reg_stab, int_stab, final_stab

In [15]:
reg_df, int_df, final_seg_df = stability_func(concatenated_df, base_week)

run_date  2024-04-07  2024-04-14  2024-04-21  2024-04-28  2024-05-05
0              False       False       False       False       False
1              False       False       False       False       False
2              False       False       False       False       False
3              False       False       False       False       False
4              False       False       False        True        True
run_date  2024-04-07  2024-04-14  2024-04-21  2024-04-28  2024-05-05
0              False       False       False       False       False
1              False       False        True        True        True
2              False        True        True        True        True
3              False       False       False       False       False
4              False       False       False       False        True
run_date  2024-04-07  2024-04-14  2024-04-21  2024-04-28  2024-05-05
0              False       False       False       False       False
1              False       False  

In [16]:
# complete flow
def stab_func_2(df, base_week, base_df, cons_type):
    
    # base_df=df[df.run_date==base_week].reset_index(drop=True)
    fdf1 = base_df.groupby(by=['city', 'run_date', cons_type], as_index=False).agg(base=('customer','nunique'))
    j = 0
    
    for i in date_list:
        
        j += 1
        if i != base_week:
            
            tdf = df[df.run_date==base_week][['run_date', 'customer', cons_type]]\
                                                .merge(df[df.run_date == i][['customer', cons_type]],on=['customer'],how='left')\
                                                .pivot_table(values='customer', 
                                                             index=['run_date',cons_type+'_x'], 
                                                             columns=cons_type+'_y', aggfunc='nunique')\
                                                .reset_index()\
                                                .fillna(0)
            
            # print(list(tdf))
            items_to_remove = ['run_date', cons_type+'_x']
            filtered_list = [x for x in list(tdf) if x not in items_to_remove]
            tdf.columns = ['run_date', cons_type]+[ j + '_' + i for j in filtered_list]
            fdf1 = fdf1.merge(tdf, on=['run_date', cons_type], how='inner')
    
    return fdf1

In [17]:
#cons_type: 'rr_regularity_final', 'rr_mean_intent_final', 'reg_intent'
finaldf = stab_func_2(concatenated_df, base_week, base_df, 'reg_intent')

In [18]:
finaldf

,city,run_date,reg_intent,base,01_daily_01_High_2024-04-14,01_daily_02_Medium_2024-04-14,02_weekly_01_High_2024-04-14,02_weekly_02_Medium_2024-04-14,02_weekly_03_Low_2024-04-14,03_bi_weekly_01_High_2024-04-14,03_bi_weekly_02_Medium_2024-04-14,03_bi_weekly_03_Low_2024-04-14,04_monthly_01_High_2024-04-14,04_monthly_02_Medium_2024-04-14,04_monthly_03_Low_2024-04-14,05_rare_need_01_High_2024-04-14,05_rare_need_02_Medium_2024-04-14,05_rare_need_03_Low_2024-04-14,01_daily_01_High_2024-04-21,01_daily_02_Medium_2024-04-21,02_weekly_01_High_2024-04-21,02_weekly_02_Medium_2024-04-21,02_weekly_03_Low_2024-04-21,03_bi_weekly_01_High_2024-04-21,03_bi_weekly_02_Medium_2024-04-21,03_bi_weekly_03_Low_2024-04-21,04_monthly_01_High_2024-04-21,04_monthly_02_Medium_2024-04-21,04_monthly_03_Low_2024-04-21,05_rare_need_01_High_2024-04-21,05_rare_need_02_Medium_2024-04-21,05_rare_need_03_Low_2024-04-21,01_daily_01_High_2024-04-28,01_daily_02_Medium_2024-04-28,02_weekly_01_High_2024-04-28,02_weekly_02_Medium_2024-04-28,02_weekly_03_Low_2024-04-28,03_bi_weekly_01_High_2024-04-28,03_bi_weekly_02_Medium_2024-04-28,03_bi_weekly_03_Low_2024-04-28,04_monthly_01_High_2024-04-28,04_monthly_02_Medium_2024-04-28,04_monthly_03_Low_2024-04-28,05_rare_need_01_High_2024-04-28,05_rare_need_02_Medium_2024-04-28,05_rare_need_03_Low_2024-04-28,01_daily_01_High_2024-05-05,01_daily_02_Medium_2024-05-05,02_weekly_01_High_2024-05-05,02_weekly_02_Medium_2024-05-05,02_weekly_03_Low_2024-05-05,03_bi_weekly_01_High_2024-05-05,03_bi_weekly_02_Medium_2024-05-05,03_bi_weekly_03_Low_2024-05-05,04_monthly_01_High_2024-05-05,04_monthly_02_Medium_2024-05-05,04_monthly_03_Low_2024-05-05,05_rare_need_01_High_2024-05-05,05_rare_need_02_Medium_2024-05-05,05_rare_need_03_Low_2024-05-05
0,Chennai,2024-04-07,01_daily_01_High,51137,46266.0,2178.0,659.0,596.0,0.0,1396.0,42.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,44400.0,2592.0,610.0,890.0,0.0,1791.0,345.0,0.0,509.0,0.0,0.0,0.0,0.0,0.0,42304.0,3079.0,644.0,1388.0,0.0,2586.0,524.0,0.0,612.0,0.0,0.0,0.0,0.0,0.0,39999.0,3453.0,653.0,1808.0,1.0,3191.0,852.0,0.0,599.0,152.0,1.0,428.0,0.0,0.0
1,Chennai,2024-04-07,01_daily_02_Medium,24218,2037.0,18152.0,17.0,2932.0,0.0,32.0,1048.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3074.0,15799.0,25.0,3496.0,11.0,110.0,1415.0,0.0,0.0,288.0,0.0,0.0,0.0,0.0,4107.0,13653.0,25.0,4022.0,35.0,167.0,1912.0,0.0,1.0,296.0,0.0,0.0,0.0,0.0,4568.0,11886.0,25.0,4475.0,39.0,243.0,2366.0,27.0,38.0,336.0,0.0,0.0,215.0,0.0
2,Chennai,2024-04-07,02_weekly_01_High,4748,603.0,12.0,2282.0,883.0,0.0,941.0,27.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,742.0,24.0,1644.0,970.0,0.0,568.0,259.0,0.0,541.0,0.0,0.0,0.0,0.0,0.0,805.0,36.0,1318.0,1085.0,0.0,561.0,350.0,0.0,592.0,1.0,0.0,0.0,0.0,0.0,813.0,42.0,1031.0,1078.0,1.0,498.0,385.0,3.0,279.0,161.0,0.0,457.0,0.0,0.0
3,Chennai,2024-04-07,02_weekly_02_Medium,97173,626.0,2666.0,718.0,78639.0,2173.0,18.0,11988.0,345.0,0.0,0.0,0.0,0.0,0.0,0.0,1205.0,4039.0,853.0,70703.0,2758.0,175.0,15235.0,826.0,0.0,1379.0,0.0,0.0,0.0,0.0,1999.0,5144.0,992.0,64900.0,3224.0,266.0,17184.0,1474.0,1.0,1983.0,6.0,0.0,0.0,0.0,2682.0,5669.0,1077.0,58432.0,3172.0,327.0,19498.0,2379.0,84.0,2564.0,148.0,0.0,1141.0,0.0
4,Chennai,2024-04-07,02_weekly_03_Low,22904,0.0,11.0,3.0,2806.0,14688.0,0.0,180.0,5216.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,22.0,7.0,3920.0,11710.0,1.0,619.0,6618.0,0.0,0.0,0.0,0.0,0.0,0.0,13.0,46.0,7.0,4794.0,9885.0,2.0,1020.0,6726.0,0.0,1.0,410.0,0.0,0.0,0.0,17.0,65.0,13.0,5102.0,7673.0,3.0,1627.0,7182.0,0.0,44.0,1178.0,0.0,0.0,0.0
5,Chennai,2024-04-07,03_bi_weekly_01_High,6439,1060.0,88.0,245.0,131.0,0.0,3883.0,684.0,2.0,311.0,35.0,0.0,0.0,0.0,0.0,1573.0,172.0,282.0,287.0,0.0,2720.0,836.0,3.0,497.0,68.0,1.0,0.0,0.0,0.0,1775.0,196.0,298.0,413.0,2.0,2042.0,948.0,11.0,606.0,142.0,3.0,3.0,0.0,0.0,1746.0,244.0,289.0,542.0,4.0,1582.0,972.0,22.0,710.0,244.0,13.0,54.0,13.0,4.0
6,Chennai,2024-04-07,03_bi_weekly_02_Medium,75690,161.0,927.0,92.0,10817.0,653.0,719.0,52341.0,3920.0,23.0,5596.0,441.0,0.0,0.0,0.0,508.0,1453.0,196.0,14256.0,1121.0,795.0,4353

In [19]:
base_df\
.groupby(by=['city_name','run_date','reg_intent'],as_index=False)\
.agg(
    user_count=('customer','nunique'),
    fe_retention_w01=('fe_flag_w01','sum'),
    fe_retention_w02=('fe_flag_w02','sum'),
    fe_retention_w03=('fe_flag_w03','sum'),
    fe_retention_w04=('fe_flag_w04','sum'),
    fe_retention_w05=('fe_flag_w05','sum'),
    fe_retention_w06=('fe_flag_w06','sum'),
    fe_retention_w07=('fe_flag_w07','sum'),
    fe_retention_w08=('fe_flag_w08','sum'),
    fe_retention_w09=('fe_flag_w09','sum'),
    fe_retention_w10=('fe_flag_w10','sum'),
    
    rr_retention_w01=('rr_flag_w01','sum'),
    rr_retention_w02=('rr_flag_w02','sum'),
    rr_retention_w03=('rr_flag_w03','sum'),
    rr_retention_w04=('rr_flag_w04','sum'),
    rr_retention_w05=('rr_flag_w05','sum'),
    rr_retention_w06=('rr_flag_w06','sum'),
    rr_retention_w07=('rr_flag_w07','sum'),
    rr_retention_w08=('rr_flag_w08','sum'),
    rr_retention_w09=('rr_flag_w09','sum'),
    rr_retention_w10=('rr_flag_w10','sum'),
        
    nr_retention_w01=('nr_flag_w01','sum'),
    nr_retention_w02=('nr_flag_w02','sum'),
    nr_retention_w03=('nr_flag_w03','sum'),
    nr_retention_w04=('nr_flag_w04','sum'),
    nr_retention_w05=('nr_flag_w05','sum'),
    nr_retention_w06=('nr_flag_w06','sum'),
    nr_retention_w07=('nr_flag_w07','sum'),
    nr_retention_w08=('nr_flag_w08','sum'),
    nr_retention_w09=('nr_flag_w09','sum'),
    nr_retention_w10=('nr_flag_w10','sum')
    )


# .to_csv(f'{month}/output_files/reg_retention_10w_newlogic.csv',index=False)

#rr_mean_intent_final, rr_regularity_final, reg_intent

# gender_tag,
# taxi_income_segment,
# ps_tag_link,
# ps_tag_auto,
# taxi_service_affinity,
# taxi_distance_preference,
# taxi_time_of_day_preference,
# taxi_day_of_week_preference

,city_name,run_date,reg_intent,user_count,fe_retention_w01,fe_retention_w02,fe_retention_w03,fe_retention_w04,fe_retention_w05,fe_retention_w06,fe_retention_w07,fe_retention_w08,fe_retention_w09,fe_retention_w10,rr_retention_w01,rr_retention_w02,rr_retention_w03,rr_retention_w04,rr_retention_w05,rr_retention_w06,rr_retention_w07,rr_retention_w08,rr_retention_w09,rr_retention_w10,nr_retention_w01,nr_retention_w02,nr_retention_w03,nr_retention_w04,nr_retention_w05,nr_retention_w06,nr_retention_w07,nr_retention_w08,nr_retention_w09,nr_retention_w10
0,Chennai,2024-04-07,01_daily_01_High,51137,45769,48456,49196,49507,49693,49826,49895,49963,50031,50084,25883,30164,32279,33471,34373,35037,35545,36071,36702,37262,23834,27874,29766,30851,31660,32304,32800,33217,33638,34029
1,Chennai,2024-04-07,01_daily_02_Medium,24218,20608,22624,23217,23451,23558,23640,23697,23734,23774,23807,10522,12689,13869,14534,15061,15430,15731,15975,16269,16583,9395,11547,12634,13261,13735,14088,14385,14623,14858,15092
2,Chennai,2024-04-07,02_weekly_01_High,4748,3199,3682,3856,3963,4022,4056,4085,4103,4119,4133,1517,1970,2194,2358,2484,2579,2653,2721,2779,2840,1346,1761,1976,2121,2238,2327,2395,2455,2497,2548
3,Chennai,2024-04-07,02_weekly_02_Medium,97173,70155,84765,89583,91564,92667,93312,93697,94019,94275,94472,29432,40520,46851,50811,53680,55818,57425,58948,60557,62068,25409,35541,41383,45093,47838,49907,51618,53108,54434,55725
4,Chennai,2024-04-07,02_weekly_03_Low,22904,13742,18234,20129,20986,21445,21731,21889,22007,22123,22202,6037,8927,10590,11664,12496,13012,13435,13852,14252,14616,5194,7804,9342,10349,11150,11681,12151,12560,12924,13230
5,Chennai,2024-04-07,03_bi_weekly_01_High,6439,4447,5264,5632,5815,5939,6004,6042,6083,6117,6138,2539,3293,3745,3999,4175,4295,4394,4507,4607,4701,2275,3012,3437,3684,3844,3972,4072,4191,4276,4354
6,Chennai,2024-04-07,03_bi_weekly_02_Medium,75690,43743,58292,64702,67958,69846,71044,71756,72329,72769,73126,17790,26368,31892,35691,38469,40613,42241,43730,45298,46680,15183,22694,27666,31166,33781,35848,37532,39016,40435,41662
7,Chennai,2024-04-07,03_bi_weekly_03_Low,62956,29278,42904,49725,53465,55833,57338,58349,59072,59687,60177,11827,19080,23897,27207,29837,31750,33295,34800,36276,37651,9962,16272,20528,23588,26047,27967,29540,31016,32323,33551
8,Chennai,2024-04-07,04_monthly_01_High,3426,1839,2297,2513,2633,2749,2814,2857,2906,2953,2986,1040,1419,1614,1746,1842,1915,1982,2042,2106,2165,959,1304,1508,1630,1732,1799,1865,1927,1972,2015
9,Chennai,2024-04-07,04_monthly_02_Medium,32923,13996,20215,23438,25379,26793,27715,28348,28887,29354,29729,5677,9034,11214,12751,14038,14941,15720,16452,17220,17904,4915,7769,9700,11121,12282,13195,13944,14652,15282,15883


In [20]:
test=list(finaldf)

In [21]:
# test.remove([['city','01_daily_2023-11-11']])

In [22]:
test=['city']+[i+'_yo' for i in test]

In [23]:
test

['city',
 'city_yo',
 'run_date_yo',
 'reg_intent_yo',
 'base_yo',
 '01_daily_01_High_2024-04-14_yo',
 '01_daily_02_Medium_2024-04-14_yo',
 '02_weekly_01_High_2024-04-14_yo',
 '02_weekly_02_Medium_2024-04-14_yo',
 '02_weekly_03_Low_2024-04-14_yo',
 '03_bi_weekly_01_High_2024-04-14_yo',
 '03_bi_weekly_02_Medium_2024-04-14_yo',
 '03_bi_weekly_03_Low_2024-04-14_yo',
 '04_monthly_01_High_2024-04-14_yo',
 '04_monthly_02_Medium_2024-04-14_yo',
 '04_monthly_03_Low_2024-04-14_yo',
 '05_rare_need_01_High_2024-04-14_yo',
 '05_rare_need_02_Medium_2024-04-14_yo',
 '05_rare_need_03_Low_2024-04-14_yo',
 '01_daily_01_High_2024-04-21_yo',
 '01_daily_02_Medium_2024-04-21_yo',
 '02_weekly_01_High_2024-04-21_yo',
 '02_weekly_02_Medium_2024-04-21_yo',
 '02_weekly_03_Low_2024-04-21_yo',
 '03_bi_weekly_01_High_2024-04-21_yo',
 '03_bi_weekly_02_Medium_2024-04-21_yo',
 '03_bi_weekly_03_Low_2024-04-21_yo',
 '04_monthly_01_High_2024-04-21_yo',
 '04_monthly_02_Medium_2024-04-21_yo',
 '04_monthly_03_Low_2024-04-2

In [24]:
finaldf.to_csv('fe_stability_v0_old.csv', index=False)

In [25]:
finaldf.head(2)

,city,run_date,reg_intent,base,01_daily_01_High_2024-04-14,01_daily_02_Medium_2024-04-14,02_weekly_01_High_2024-04-14,02_weekly_02_Medium_2024-04-14,02_weekly_03_Low_2024-04-14,03_bi_weekly_01_High_2024-04-14,03_bi_weekly_02_Medium_2024-04-14,03_bi_weekly_03_Low_2024-04-14,04_monthly_01_High_2024-04-14,04_monthly_02_Medium_2024-04-14,04_monthly_03_Low_2024-04-14,05_rare_need_01_High_2024-04-14,05_rare_need_02_Medium_2024-04-14,05_rare_need_03_Low_2024-04-14,01_daily_01_High_2024-04-21,01_daily_02_Medium_2024-04-21,02_weekly_01_High_2024-04-21,02_weekly_02_Medium_2024-04-21,02_weekly_03_Low_2024-04-21,03_bi_weekly_01_High_2024-04-21,03_bi_weekly_02_Medium_2024-04-21,03_bi_weekly_03_Low_2024-04-21,04_monthly_01_High_2024-04-21,04_monthly_02_Medium_2024-04-21,04_monthly_03_Low_2024-04-21,05_rare_need_01_High_2024-04-21,05_rare_need_02_Medium_2024-04-21,05_rare_need_03_Low_2024-04-21,01_daily_01_High_2024-04-28,01_daily_02_Medium_2024-04-28,02_weekly_01_High_2024-04-28,02_weekly_02_Medium_2024-04-28,02_weekly_03_Low_2024-04-28,03_bi_weekly_01_High_2024-04-28,03_bi_weekly_02_Medium_2024-04-28,03_bi_weekly_03_Low_2024-04-28,04_monthly_01_High_2024-04-28,04_monthly_02_Medium_2024-04-28,04_monthly_03_Low_2024-04-28,05_rare_need_01_High_2024-04-28,05_rare_need_02_Medium_2024-04-28,05_rare_need_03_Low_2024-04-28,01_daily_01_High_2024-05-05,01_daily_02_Medium_2024-05-05,02_weekly_01_High_2024-05-05,02_weekly_02_Medium_2024-05-05,02_weekly_03_Low_2024-05-05,03_bi_weekly_01_High_2024-05-05,03_bi_weekly_02_Medium_2024-05-05,03_bi_weekly_03_Low_2024-05-05,04_monthly_01_High_2024-05-05,04_monthly_02_Medium_2024-05-05,04_monthly_03_Low_2024-05-05,05_rare_need_01_High_2024-05-05,05_rare_need_02_Medium_2024-05-05,05_rare_need_03_Low_2024-05-05
0,Chennai,2024-04-07,01_daily_01_High,51137,46266.0,2178.0,659.0,596.0,0.0,1396.0,42.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,44400.0,2592.0,610.0,890.0,0.0,1791.0,345.0,0.0,509.0,0.0,0.0,0.0,0.0,0.0,42304.0,3079.0,644.0,1388.0,0.0,2586.0,524.0,0.0,612.0,0.0,0.0,0.0,0.0,0.0,39999.0,3453.0,653.0,1808.0,1.0,3191.0,852.0,0.0,599.0,152.0,1.0,428.0,0.0,0.0
1,Chennai,2024-04-07,01_daily_02_Medium,24218,2037.0,18152.0,17.0,2932.0,0.0,32.0,1048.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3074.0,15799.0,25.0,3496.0,11.0,110.0,1415.0,0.0,0.0,288.0,0.0,0.0,0.0,0.0,4107.0,13653.0,25.0,4022.0,35.0,167.0,1912.0,0.0,1.0,296.0,0.0,0.0,0.0,0.0,4568.0,11886.0,25.0,4475.0,39.0,243.0,2366.0,27.0,38.0,336.0,0.0,0.0,215.0,0.0


In [26]:
finaldf.to_clipboard(index=False)

## fe_regularity_final

In [30]:
#cons_type: 'rr_regularity_final', 'rr_mean_intent_final', 'reg_intent'
finaldf_v2 = stab_func_2(concatenated_df, base_week, base_df, 'fe_regularity_final')

In [31]:
finaldf_v2

,city,run_date,fe_regularity_final,base,01_daily_2024-04-14,02_weekly_2024-04-14,03_bi_weekly_2024-04-14,04_monthly_2024-04-14,05_rare_need_2024-04-14,01_daily_2024-04-21,02_weekly_2024-04-21,03_bi_weekly_2024-04-21,04_monthly_2024-04-21,05_rare_need_2024-04-21,01_daily_2024-04-28,02_weekly_2024-04-28,03_bi_weekly_2024-04-28,04_monthly_2024-04-28,05_rare_need_2024-04-28,01_daily_2024-05-05,02_weekly_2024-05-05,03_bi_weekly_2024-05-05,04_monthly_2024-05-05,05_rare_need_2024-05-05
0,Chennai,2024-04-07,01_daily,75355,68633.0,4204.0,2518.0,0.0,0.0,65865.0,5032.0,3661.0,797.0,0.0,63143.0,6114.0,5189.0,909.0,0.0,59906.0,7001.0,6679.0,1126.0,643.0
1,Chennai,2024-04-07,02_weekly,124825,3918.0,102192.0,18715.0,0.0,0.0,6039.0,92565.0,24301.0,1920.0,0.0,8043.0,86205.0,27583.0,2994.0,0.0,9288.0,77579.0,31902.0,4458.0,1598.0
2,Chennai,2024-04-07,03_bi_weekly,145085,2242.0,17722.0,110125.0,14976.0,0.0,3722.0,24178.0,96557.0,20570.0,0.0,4819.0,28563.0,85868.0,25695.0,24.0,5443.0,32012.0,75642.0,30498.0,1249.0
3,Chennai,2024-04-07,04_monthly,81887,0.0,0.0,22507.0,53367.0,4268.0,0.0,0.0,27973.0,43815.0,7285.0,334.0,439.0,30743.0,36807.0,9556.0,1012.0,2618.0,30270.0,31209.0,11595.0
4,Chennai,2024-04-07,05_rare_need,17725,0.0,0.0,0.0,6935.0,7882.0,0.0,0.0,670.0,7382.0,5610.0,0.0,0.0,1891.0,6629.0,4435.0,0.0,0.0,3060.0,5955.0,3581.0


In [32]:
finaldf_v2.to_csv('fe_stability_v0_old_agg.csv', index=False)

In [33]:
finaldf_v2.to_clipboard(index=False)